- These codes splits the digital archived copies into page units. 

- The codes are executed in the following order.
    1. Remove redundant spaces around a digital copies.
    
        <=Removes the areas of the image covered with while color.
        
    2. Cuts the page at the border of the two pages.
        
        Borders are detected by searching for a black ertical line.

Code 1: Define the functions and install packages

In [1]:
import matplotlib.pyplot as plt
from scipy.signal import find_peaks
import numpy as np
import cv2
import os
import pandas as pd

def Split(Page):    
    img = cv2.imread(path+"Page"+Page+".jpg")
    hh, ww = img.shape[:2]
    # convert to grayscale
    gray = cv2.cvtColor(img,cv2.COLOR_BGR2GRAY)
    # threshold gray image
    thresh = cv2.threshold(gray, 200, 255, cv2.THRESH_BINARY)[1]

    # count number of non-zero pixels in each row. 
    countCol = np.count_nonzero(thresh, axis=0)
    countRow = np.count_nonzero(thresh, axis=1)

    ### Find location of biggest jumps/falls.
    ###Top and Bottom
    peaks, _ = find_peaks(countRow, distance=15)
    HeightTop=abs(np.diff(countRow[peaks][peaks<hh//5]))
    HeightBtm=abs(np.diff(countRow[peaks][peaks>4*hh//5]))
    max1Height=np.sort(HeightTop)[-1]
    max2Height=np.sort(HeightBtm)[-1]
    X=peaks[ : -1][abs(np.diff(countRow[peaks]))==max1Height][0]
    Y=peaks[ : -1][abs(np.diff(countRow[peaks]))==max2Height][0]
    Top=min([X,Y])
    Btm=max([X,Y])

    ###Left and Right
    peaks, _ = find_peaks(countCol, distance=30)
    HeightLeft=abs(np.diff(countCol[peaks][peaks<ww//5]))
    HeightRight=abs(np.diff(countCol[peaks][peaks>4*ww//5]))
    max1Height=np.sort(HeightLeft)[-1]
    max2Height=np.sort(HeightRight)[-1]
    V=peaks[ : -1][abs(np.diff(countCol[peaks]))==max1Height][0]
    Z=peaks[ : -1][abs(np.diff(countCol[peaks]))==max2Height][0]
    Left=min([V,Z])
    Right=max([V,Z])

    cropped=img[Top:Btm, Left:Right] 
    #cv2.imshow("Result", cropped)
    #cv2.waitKey(0)
    
    #Code for cutting image Side-by-Side
    # read image+crop image
    crop_img=img[Top:Btm, Left:Right]
    HH, WW = crop_img.shape[:2]
    # convert to grayscale
    gray = cv2.cvtColor(crop_img,cv2.COLOR_BGR2GRAY)
    # threshold gray image
    thresh = cv2.threshold(gray, 200, 255, cv2.THRESH_BINARY)[1]

    # count number of non-zero pixels in each column and row. 
    countCol = np.count_nonzero(thresh, axis=0)
    countRow = np.count_nonzero(thresh, axis=1)

    ### This finds the height of the lowest valley
    peaks, _ = find_peaks(-countCol, distance=15)
    maxHeight=np.sort(abs(np.diff(countCol[peaks][(peaks<4*WW//7) & (peaks>3*WW//7)])))[-1]
    Border=peaks[ : -1][abs(np.diff(countCol[peaks]))==maxHeight][0]

    ### loop over contours and get bounding boxes and ycenter and draw horizontal line at ycenter
    result = crop_img.copy()
    LeftImage=result[0:HH,0:Border]
    RighImage=result[0:HH,Border:WW]
    #cv2.imshow("LeftImage", LeftImage)
    #cv2.imshow("RightImage", RighImage)
    #cv2.waitKey(0)

    return(list((LeftImage,RighImage)))

Code 2: Define Years

In [2]:
Year=1939

Code 3: We need to create a seperate directory for each page of the archived copy.
Check the starting and ending page of the staff directory and define a variable with the numbers.

In [5]:
StrPage=40
EndPage=294

#Set directory path
save_path="C:\\Users\\Keitaro Ninomiya\\Box\\Research Notes (keitaro2@illinois.edu)\\Tokyo_Jobs\\Raw_Data\\Splited\\"+str(Year)+"\\"

#Create directory for each page
for Num in range(StrPage,EndPage+1):
    Page="Page"+"{:03d}".format(2*(Num-StrPage))
    path2 = os.path.join(save_path, Page)
    if not os.path.exists(path2):
        os.mkdir(path2)
    
    Page="Page"+"{:03d}".format(2*(Num-StrPage)+1)
    path2 = os.path.join(save_path, Page)
    if not os.path.exists(path2):
        os.mkdir(path2)

Code 4: Splitting copies into pages.
Each page is saved into the corresponding directory that we made on Code 2.

In [6]:
# read image+crop image
import os
FailedList=[]
path=r'C:\\Users\\Keitaro Ninomiya\\Box\\Research Notes (keitaro2@illinois.edu)\\Tokyo_Jobs/Raw_Data\\Metropolitan_DA\\'+str(Year)+'\\Line\\'
for Num in range(StrPage,EndPage):
    try:
        print(Num)
        Righ=Split("{:03d}".format(Num))[1]
        Left=Split("{:03d}".format(Num))[0]

        Page="Page"+"{:03d}".format(2*(Num-StrPage))
        path2 = os.path.join(save_path, Page)
        cv2.imwrite(path2+"\\"+Page+".jpg", Righ)

        Page="Page"+"{:03d}".format(2*(Num-StrPage)+1)
        path2 = os.path.join(save_path, Page)
        cv2.imwrite(path2+"\\"+Page+".jpg", Left)
    except:
        print("Error at Page " +str(Num))
        FailedList.append(Num)
        continue
    

40
41
42
43
44
45
46
47
48
49
50
51
52
53
54
55
56
57
58
59
60
61
62
63
64
65
66
67
68
69
70
71
72
73
74
75
76
77
78
79
80
81
82
83
84
85
86
87
88
89
90
91
92
93
94
95
96
97
98
99
100
101
102
103
104
105
106
107
108
109
110
111
112
113
114
115
116
117
118
119
120
121
122
123
124
125
126
127
128
129
130
131
132
133
134
135
136
137
138
139
140
141
142
143
144
145
146
147
148
149
150
151
152
153
154
155
156
157
158
159
160
161
162
163
164
165
166
167
168
169
170
171
172
173
174
175
176
177
178
179
180
181
182
183
184
185
186
187
188
189
190
191
192
193
194
195
196
197
198
199
200
201
202
203
204
205
206
207
208
209
210
211
212
213
214
215
216
217
218
219
220
221
222
223
224
225
226
227
228
229
230
231
232
233
234
235
236
237
238
239
240
241
242
243
244
245
246
247
248
249
250
251
252
253
254
255
256
257
258
259
260
261
262
263
264
265
266
267
268
269
270
271
272
273
274
275
276
277
278
279
280
281
282
283
284
285
286
287
288
289
290
291
292
293


In [10]:
FailRate=len(FailedList)/(EndPage-StrPage)
if FailRate<0.2:
    print('Fantastic!! Success Rate is '+str(1-FailRate))
else:
    print('残念...Success Rate is '+str(1-FailRate))
DF=pd.read_csv('C:\\Users\\Keitaro Ninomiya\\Box\\Research Notes (keitaro2@illinois.edu)\\Tokyo_Jobs\\Processed_Data\\Records.csv',index_col=False)
DF.loc[int(Year)-1934, 'Page'] = 1-FailRate
display(DF)
DF.to_csv('C:\\Users\\Keitaro Ninomiya\\Box\\Research Notes (keitaro2@illinois.edu)\\Tokyo_Jobs\\Processed_Data\\Records.csv',index=False)

Fantastic!! Success Rate is 1.0


,Year,WritePage,Page,PageFin,Office,OfficeFin,Unit,UnitFin,Position,PositionFin,Data
0,1934,1,0.985714286,.,0.9,.,NaN,.,0.08,.,.
1,1935,1,1,.,0.984375,.,NaN,.,-0.09375,.,.
2,1936,0.235294118,.,.,0.235294118,.,NaN,.,.,.,.
3,1937,1,1,.,0.865671642,.,NaN,.,.,.,.
4,1938,1,1,.,1,.,0.988095238,.,0.98816568,.,0.619957537
5,1939,1,1.0,.,1.0,.,.,.,.,.,.
6,1940,1,.,.,0.991803279,.,NaN,.,.,.,.
7,1941,1,1,.,.,.,NaN,.,.,.,.
8,1942,1,1,.,.,.,NaN,.,.,.,.
9,1943,1,.,.,.,.,.,.,.,.,.
